# EDA 02: Seasonality, Day-of-Week, and Holiday Effect Analysis

This notebook evaluates seasonal demand cycles across days of the week, months of the year, and public holidays using `calendar_dim`.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

df_sales = pd.read_parquet('data/processed/sales_fact.parquet')
df_cal = pd.read_parquet('data/processed/calendar_dim.parquet')

# Merge sales_fact with calendar_dim on date
df_merged = df_sales.merge(df_cal, on='date', how='inner')
print(f"Merged Dataset Rows: {len(df_merged):,}")
df_merged.head()


Merged Dataset Rows: 850,892


In [2]:
# Day-of-Week Seasonality (0=Monday, 6=Sunday)
dow_names = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri', 5: 'Sat', 6: 'Sun'}
df_merged['dow_name'] = df_merged['day_of_week'].map(dow_names)

dow_summary = df_merged.groupby(['day_of_week', 'dow_name']).agg(
    total_revenue=('total_sales', 'sum'),
    avg_daily_revenue=('total_sales', 'mean'),
    transaction_count=('sales_id', 'count')
).reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(data=dow_summary, x='dow_name', y='avg_daily_revenue', palette='Blues_d')
plt.title('Average Revenue per Transaction by Day of Week')
plt.xlabel('Day of Week')
plt.ylabel('Mean Transaction Revenue ($)')
plt.tight_layout()
plt.show()


In [3]:
# Monthly Seasonality (Month 1 to 12)
month_names = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun', 7:'Jul', 8:'Aug', 9:'Sep', 10:'Oct', 11:'Nov', 12:'Dec'}
df_merged['month_name'] = df_merged['month'].map(month_names)

monthly_agg = df_merged.groupby(['month', 'month_name']).agg(
    total_revenue=('total_sales', 'sum'),
    mean_revenue=('total_sales', 'mean'),
    transaction_count=('sales_id', 'count')
).reset_index()

plt.figure(figsize=(12, 5))
sns.lineplot(data=monthly_agg, x='month_name', y='total_revenue', marker='o', color='crimson', linewidth=2.5)
plt.title('Total Revenue Seasonality by Month of Year')
plt.xlabel('Month')
plt.ylabel('Total Revenue ($)')
plt.tight_layout()
plt.show()


In [4]:
# Weekend vs Weekday Sales Analysis
weekend_summary = df_merged.groupby('is_weekend').agg(
    total_revenue=('total_sales', 'sum'),
    mean_revenue=('total_sales', 'mean'),
    transaction_count=('sales_id', 'count')
).reset_index()

weekend_summary['type'] = np.where(weekend_summary['is_weekend'], 'Weekend', 'Weekday')
print(weekend_summary)

plt.figure(figsize=(8, 5))
sns.barplot(data=weekend_summary, x='type', y='mean_revenue', palette='Set2')
plt.title('Average Order Revenue: Weekday vs Weekend')
plt.ylabel('Mean Sales ($)')
plt.show()


   is_weekend  total_revenue  mean_revenue  transaction_count     type
0       False   1.168727e+10  16619.913498             703209  Weekday
1        True   8.749373e+08   5924.427875             147683  Weekend


In [5]:
# Holiday Effect Analysis (Holiday vs Non-Holiday)
holiday_summary = df_merged.groupby('is_holiday').agg(
    total_revenue=('total_sales', 'sum'),
    mean_sales=('total_sales', 'mean'),
    median_sales=('total_sales', 'median'),
    transaction_count=('sales_id', 'count')
).reset_index()

holiday_summary['type'] = np.where(holiday_summary['is_holiday'], 'Holiday Day', 'Regular Day')
print(holiday_summary)

reg_mean = holiday_summary.loc[~holiday_summary['is_holiday'], 'mean_sales'].values[0]
hol_mean = holiday_summary.loc[holiday_summary['is_holiday'], 'mean_sales'].values[0]
lift_pct = ((hol_mean - reg_mean) / reg_mean) * 100
print(f"Holiday Revenue Lift: {lift_pct:.2f}%")


   is_holiday  total_revenue  ...  transaction_count         type
0       False   1.239569e+10  ...             845508  Regular Day
1        True   1.665239e+08  ...               5384  Holiday Day

[2 rows x 6 columns]
Holiday Revenue Lift: 110.97%
